# Random forests, end to end

The same dataset and the same split as the logistic regression notebook, so the two scores can be compared directly.

Two things are deliberately missing: there is no scaler, and there is no penalty to tune. A tree splits on thresholds, so multiplying a feature by a thousand changes nothing about where the split lands. That is most of why forests are the default first move on messy tabular data.

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import cross_val_score, train_test_split

# 1. Same dataset as the logistic regression notebook, so the scores compare
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. No scaling. A tree splits on thresholds, so units do not matter.
model = RandomForestClassifier(
    n_estimators=500,
    oob_score=True,      # free validation from the rows each tree never saw
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

# 3. Three different estimates of how good it is
print(f"OOB score:      {model.oob_score_:.4f}")
print(f"5-fold CV:      {cross_val_score(model, X_train, y_train, cv=5).mean():.4f}")
y_pred = model.predict(X_test)
print(f"Test accuracy:  {accuracy_score(y_test, y_pred):.4f}")

print("\nConfusion matrix (rows = actual, cols = predicted):")
print(confusion_matrix(y_test, y_pred))

# 4. Which features the forest leaned on
importances = sorted(zip(model.feature_importances_, data.feature_names), reverse=True)
print("\nTop 5 features by impurity importance:")
for score, name in importances[:5]:
    print(f"  {score:.4f}  {name}")

OOB score:      0.9626
5-fold CV:      0.9604
Test accuracy:  0.9649

Confusion matrix (rows = actual, cols = predicted):
[[40  3]
 [ 1 70]]

Top 5 features by impurity importance:
  0.1397  worst concave points
  0.1224  worst area
  0.1153  mean concave points
  0.1144  worst perimeter
  0.0825  worst radius

## What the output is telling you

- **Three estimates, three slightly different numbers** — OOB 0.9626, 5-fold CV 0.9604, test 0.9649. They agree here, which is the reassuring case. When they disagree sharply, trust the one whose split matches how the model will actually be used.
- **Out-of-bag scoring is close to free.** Each tree trains on a bootstrap sample and about a third of the rows are left out of it; scoring each row using only the trees that never saw it gives a validation estimate without a separate split.
- **The confusion matrix says what accuracy hides.** Three malignant cases were classified benign, one benign was called malignant. Those two mistakes have very different costs.
- **The forest scored 0.9649 against logistic regression's 0.9737.** The more flexible model lost. On 455 clean, well-separated rows there was no non-linear structure for it to find, and the extra variance cost it.

## When to reach for this

Random forests earn their place on tabular data with mixed types, non-linear relationships, outliers and missing-ish values — the situation where a linear model needs a lot of manual feature work first. They need almost no preprocessing and are hard to make catastrophically wrong.

Reach elsewhere when you need calibrated probabilities (forests are over-confident near the edges), when the model must be explainable line by line, or when the data is small and linearly separable, as it is here.

## Extend this notebook

- Treat impurity importance with suspicion: it inflates high-cardinality and continuous features. Compare it against `permutation_importance`, which measures what actually degrades when a feature is shuffled.
- Set `max_depth` and watch the gap between training and OOB score close.
- Drop `n_estimators` to 10 and see the variance in the score across seeds.